# Virtual Environments

Every professional Python project lives inside its own isolated environment. This is not optional — it is a fundamental habit that separates a disciplined developer from someone whose machine breaks every six months. This module teaches you how virtual environments work and how to use them correctly.

---

## Table of Contents

- [ ] 1. What is a Virtual Environment?
- [ ] 2. Creating a Virtual Environment
- [ ] 3. Activating and Deactivating
- [ ] 4. Installing Packages Inside the Environment
- [ ] 5. requirements.txt — Sharing Your Setup
- [ ] 6. Connecting Jupyter to Your Environment
- [ ] 7. Summary


---

## 1. What is a Virtual Environment?

**Analogy:** Imagine you work on two security projects at the same time. Project A is a legacy intrusion detection system that needs an old version of the `requests` library (version 2.20). Project B is a new threat intelligence tool that needs the latest version (2.31). If you install both on your machine globally, they overwrite each other and at least one project breaks.

A **virtual environment** solves this by giving each project its own private box of installed packages — completely isolated from the system Python and from every other project.

```
Without venv:                         With venv:

[ System Python ]                     [ System Python ]  (untouched)
  |- requests 2.31  <-- conflict!       |
  |- scapy 2.5                          |-- [ .venv for project A ]
  |- pandas 1.0                         |     |- requests 2.20
  |- ... everything mixed together      |
                                         |-- [ .venv for project B ]
                                               |- requests 2.31
                                               |- scapy 2.5
```

### The rule

**One project = one virtual environment.** Always. No exceptions.

This applies whether you are writing a 10-line analysis script or a production SOC automation platform.


In [ ]:
# You can inspect the current Python installation from inside Python
# This tells you WHERE Python is running from right now
import sys

print("Python executable:", sys.executable)
print("Python version:   ", sys.version)
print("Install prefix:   ", sys.prefix)

# If you are inside a virtual environment, sys.prefix will point to the .venv folder
# If you are using the system Python, it will point to a system-level path


In [ ]:
# Check if the current session is inside a virtual environment
import sys
import os

# sys.base_prefix is the original system prefix.
# sys.prefix is the active prefix (venv overrides this).
# If they are different, you are inside a virtual environment.

in_venv = sys.prefix != sys.base_prefix

if in_venv:
    print("[OK] Running inside a virtual environment")
    print("     Venv location:", sys.prefix)
else:
    print("[WARNING] Running with the SYSTEM Python")
    print("          Create and activate a venv before installing packages")


### Check Your Understanding: Virtual Environments

**Predict** — Run the detection cell above. Are you currently inside a virtual environment? What does your output look like?

If you are using the system Python, that is expected — you will fix that in Section 2.


---

## 2. Creating a Virtual Environment

**Analogy:** Creating a virtual environment is like building a new, empty kitchen for a specific project. Right now it has the essentials (Python, pip) but nothing else. You will stock it with exactly what the project needs.

The `venv` module is part of the Python Standard Library — it is available on every Python 3.3+ installation with no extra install needed.

### The creation command

Run this in your **terminal** (not in a Jupyter cell), from inside your project folder:

```bash
python -m venv .venv
```

Breaking this down:

| Part | Meaning |
|------|---------|
| `python -m` | Run a module as a script |
| `venv` | The module that creates isolated environments |
| `.venv` | The folder to create (convention: name it `.venv`) |

> **Convention:** Name your virtual environment folder `.venv` (with the dot). The dot makes it a hidden folder on Linux/macOS — it stays out of your way visually. Some teams use `venv` without the dot. Both work; `.venv` is the modern standard.

### What gets created?

```
your_project/
  |- .venv/              <-- the virtual environment folder
  |    |- bin/           <-- Python interpreter and scripts (Linux/macOS)
  |    |   or Scripts/   <-- same, but on Windows
  |    |- lib/           <-- installed packages go here
  |    |- include/       <-- C headers for packages with C extensions
  |    |- pyvenv.cfg     <-- configuration file pointing to the system Python
  |
  |- your_script.py
  |- requirements.txt
```

> **Important:** Never commit the `.venv` folder to Git. Add it to your `.gitignore` file. The environment can always be recreated from `requirements.txt`.


In [ ]:
# You can also create a virtual environment using Python code
# (Useful for automated setup scripts)
import venv
import pathlib

# Create a demo environment in a temp location
env_path = pathlib.Path("/tmp/demo_venv")

if not env_path.exists():
    venv.create(env_path, with_pip=True)   # with_pip=True ensures pip is included
    print(f"[OK] Virtual environment created at: {env_path}")
else:
    print(f"[OK] Environment already exists at: {env_path}")

# List what was created
for item in sorted(env_path.iterdir()):
    print(f"  {item.name}/")


### Practice - Section 2

**Write** — In your terminal, navigate to a folder of your choice and create a virtual environment named `.venv`. Then use `pathlib` in the cell below to verify that the `.venv` folder was created and list its contents.


In [ ]:
# Your code here
# Update the path below to point to where you created your .venv
import pathlib

venv_path = pathlib.Path(".venv")   # adjust if needed

# Check if it exists and list contents


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import pathlib

venv_path = pathlib.Path(".venv")

if venv_path.exists():
    print(f"[OK] .venv found at: {venv_path.resolve()}")
    print("Contents:")
    for item in sorted(venv_path.iterdir()):
        print(f"  {item.name}/")
else:
    print("[NOT FOUND] .venv does not exist here.")
    print("Run this in your terminal first:")
    print("  python -m venv .venv")
```

</details>

---

## 3. Activating and Deactivating

**Analogy:** Creating a virtual environment only builds the kitchen. **Activating** it means walking in and starting to work there. Until you activate, your terminal still uses the system Python — creating the folder alone does nothing.

Activation commands are run in the **terminal**, not in Python:

| OS | Command |
|----|--------|
| **Linux / macOS** | `source .venv/bin/activate` |
| **Windows (PowerShell)** | `.venv\Scripts\Activate.ps1` |
| **Windows (CMD)** | `.venv\Scripts\activate.bat` |

### How to know you are activated

When a virtual environment is active, your terminal prompt shows its name in parentheses:

```bash
# Before activation:
user@host:~/projects/soc-tool$

# After activation:
(.venv) user@host:~/projects/soc-tool$
```

### Deactivating

To leave the virtual environment and return to the system Python:

```bash
deactivate
```

This works on all operating systems.

### Warning — Closing the terminal deactivates automatically

Activation is **per-session**. Closing your terminal exits the environment. Every time you open a new terminal to work on a project, you must re-activate the environment.


In [ ]:
# After activating, verify you are in the right environment
# Run this AFTER activating .venv in your terminal and connecting Jupyter to it

import sys

in_venv = sys.prefix != sys.base_prefix
venv_name = sys.prefix.split("/")[-1] if "/" in sys.prefix else sys.prefix.split("\\")[-1]

if in_venv:
    print(f"[OK] Virtual environment active: {venv_name}")
    print(f"     Python: {sys.executable}")
else:
    print("[WARNING] No virtual environment active.")
    print("          Activate it before installing packages.")


### Practice - Section 3

**Write** — Using the terminal, create a new project folder called `threat_intel`. Inside it, create and activate a virtual environment. Then verify the activation by running the detection cell above.

Document the exact terminal commands you used below (in the markdown cell — double-click to edit).


**My terminal commands:**

```bash
# Write your commands here
```


---

## 4. Installing Packages Inside the Environment

**Analogy:** Once the kitchen is built and you are standing in it (activated), you can stock it. `pip install` puts packages into the active environment only — they go nowhere near the system Python.

### The basic workflow

```bash
# 1. Make sure the environment is active (you see (.venv) in the prompt)
(.venv) $ pip install requests

# 2. Install a specific version
(.venv) $ pip install requests==2.31.0

# 3. See what is installed in this environment
(.venv) $ pip list

# 4. Remove a package
(.venv) $ pip uninstall requests
```

### Checking what is installed from Python

Sometimes you need to inspect the installed packages programmatically — for example, to verify that a deployment has all required dependencies.


In [ ]:
# List all packages installed in the active environment
import subprocess
import sys

# Use the same Python that is running this notebook to call pip
result = subprocess.run(
    [sys.executable, "-m", "pip", "list", "--format=columns"],
    capture_output=True,
    text=True
)

print(result.stdout)


In [ ]:
# Check if a specific package is installed and get its version
# This pattern is useful in deployment scripts and health checks

import importlib.metadata

packages_to_check = ["requests", "scapy", "pandas", "pip"]  # edit this list

for pkg_name in packages_to_check:
    try:
        version = importlib.metadata.version(pkg_name)
        print(f"[OK]      {pkg_name:<15} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"[MISSING] {pkg_name:<15} not installed")


### Warning — Never run pip without an active environment

If you run `pip install` without an active virtual environment, packages go into the system Python. This is how dependency conflicts and broken system tools happen.

**Before every `pip install`, check:** does my prompt show `(.venv)`?

Some teams enforce this with `pip`'s `--require-virtualenv` flag or the environment variable `PIP_REQUIRE_VIRTUALENV=1`.


### Practice - Section 4

**Write** — With your `.venv` active, install the `requests` package from the terminal. Then run the cell below to verify it is installed and print its version.


In [ ]:
# Your code here: check whether requests is installed and print its version


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import importlib.metadata

try:
    version = importlib.metadata.version("requests")
    print(f"[OK] requests version {version} is installed")
except importlib.metadata.PackageNotFoundError:
    print("[MISSING] requests is not installed.")
    print("Run: pip install requests")
```

</details>

---

## 5. requirements.txt — Sharing Your Setup

**Analogy:** A `requirements.txt` is the recipe for your environment. If you give a colleague your code but not the installed packages (which you never commit to Git), they need a way to recreate your exact setup. The requirements file is that way — one file that lists every package and its version, so anyone can rebuild your environment from scratch in one command.

### Generating the file

With your environment active:

```bash
(.venv) $ pip freeze > requirements.txt
```

`pip freeze` prints every installed package with its exact pinned version. The `>` redirects that output into a file.

### Installing from the file

```bash
# After creating and activating a fresh .venv:
(.venv) $ pip install -r requirements.txt
```

The `-r` flag means "read from file".

### What a requirements.txt looks like

```
certifi==2024.2.2
charset-normalizer==3.3.2
idna==3.7
requests==2.31.0
urllib3==2.2.1
```

The `==` means **pinned to this exact version**. This ensures reproducible installs — the same code runs with the same library behaviour on every machine.


In [ ]:
# Generate requirements content programmatically (same as pip freeze)
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True
)

print("Current environment requirements:")
print("-" * 40)
print(result.stdout if result.stdout else "(no third-party packages installed)")


In [ ]:
# Read and parse a requirements.txt file in Python
# Useful for CI/CD scripts that validate dependencies before deployment

requirements_content = """requests==2.31.0
scapy==2.5.0
python-nmap==1.6.0
paramiko==3.4.0
"""

# Parse the file content into a list of (package, version) tuples
dependencies = []
for line in requirements_content.strip().splitlines():
    line = line.strip()
    if line and not line.startswith("#"):   # skip blanks and comments
        if "==" in line:
            name, version = line.split("==")
            dependencies.append((name.strip(), version.strip()))

print(f"Found {len(dependencies)} dependencies:")
for name, version in dependencies:
    print(f"  {name:<20} {version}")


### Practice - Section 5

**Write** — With your `.venv` active in the terminal, run `pip freeze > requirements.txt`. Then use the cell below to read the file, count how many packages are installed, and print the package names.


In [ ]:
# Your code here
# Read requirements.txt, count packages, print package names only (without versions)


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
from pathlib import Path

req_file = Path("requirements.txt")

if not req_file.exists():
    print("requirements.txt not found. Run: pip freeze > requirements.txt")
else:
    packages = []
    for line in req_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            name = line.split("==")[0].strip()   # take only the name, drop version
            packages.append(name)

    print(f"Total packages installed: {len(packages)}")
    print()
    for pkg in sorted(packages):
        print(f"  {pkg}")
```

</details>

---

## 6. Connecting Jupyter to Your Environment

**Analogy:** Jupyter Notebooks run using a **kernel** — the Python engine in the background. By default, the kernel uses whatever Python launched Jupyter. If you want a notebook to use your `.venv` packages, you must register the virtual environment as a Jupyter kernel.

### Step-by-step setup

**With your `.venv` active** in the terminal:

**Step 1 — Install ipykernel inside the venv**
```bash
(.venv) $ pip install ipykernel
```

**Step 2 — Register the venv as a Jupyter kernel**
```bash
(.venv) $ python -m ipykernel install --user --name=.venv --display-name "Python (.venv)"
```

**Step 3 — Select the kernel in Jupyter**

In the Jupyter interface: **Kernel > Change Kernel > Python (.venv)**

Or in VS Code: click the kernel selector in the top-right corner.

### Verifying the connection

After switching kernels, run the cell below. It should print the path inside your `.venv` folder.


In [2]:
# Verify which kernel is running this notebook
import sys

print("Kernel Python path:")
print(" ", sys.executable)
print()

in_venv = sys.prefix != sys.base_prefix
if in_venv:
    print("[OK] This notebook is running inside a virtual environment")
    print("     Environment:", sys.prefix)
else:
    print("[WARNING] This notebook is using the system Python kernel")
    print("          Follow Section 6 steps to switch to the .venv kernel")


Kernel Python path:
  c:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv\Scripts\python.exe

[OK] This notebook is running inside a virtual environment
     Environment: c:\Users\Zoltan\python-cybersecurity-progress\threat_intel\.venv


### Listing available Jupyter kernels

You can see all registered kernels from the terminal:

```bash
jupyter kernelspec list
```

To remove a kernel you no longer need:

```bash
jupyter kernelspec remove .venv
```


In [ ]:
# List available Jupyter kernels from Python
import subprocess

result = subprocess.run(
    ["jupyter", "kernelspec", "list"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print("Could not list kernels. Is Jupyter installed?")
    print("Run: pip install jupyter")


### Practice - Section 6

**Write** — Register your `.venv` as a Jupyter kernel using the terminal commands in this section. Then switch this notebook to that kernel and run the verification cell above. Confirm it shows `[OK]`.


---

## 7. Summary

| Concept | Key rule |
|---------|----------|
| **Virtual environment** | An isolated Python installation with its own packages. One project = one environment. |
| **venv module** | Built into Python 3.3+. `python -m venv .venv` creates the environment. |
| **Activation** | `source .venv/bin/activate` (Linux/macOS) or `.venv\Scripts\Activate.ps1` (Windows). Activation is per terminal session. |
| **Deactivation** | `deactivate` — works on all OSes. |
| **pip inside venv** | Only installs into the active environment. Always verify `(.venv)` in your prompt before running `pip install`. |
| **requirements.txt** | `pip freeze > requirements.txt` captures your environment. `pip install -r requirements.txt` recreates it. |
| **Git** | Never commit `.venv/`. Add it to `.gitignore`. Commit `requirements.txt` instead. |
| **Jupyter kernel** | `pip install ipykernel` + `python -m ipykernel install --user --name=.venv` registers the venv as a selectable kernel. |
| **sys.prefix** | `sys.prefix != sys.base_prefix` is True when running inside a virtual environment. |

### Commands to remember

| Context | Command | Purpose |
|---------|---------|--------|
| Terminal | `python -m venv .venv` | Create environment |
| Terminal | `source .venv/bin/activate` | Activate (Linux/macOS) |
| Terminal | `.venv\Scripts\Activate.ps1` | Activate (Windows PS) |
| Terminal | `deactivate` | Deactivate |
| Terminal | `pip install package` | Install into active env |
| Terminal | `pip freeze > requirements.txt` | Export environment |
| Terminal | `pip install -r requirements.txt` | Recreate environment |
| Terminal | `pip install ipykernel` | Enable Jupyter support |

---

### Next Steps

You can now create, manage, and share isolated Python environments correctly.

**-> [01_Advanced_VirtualEnv.ipynb](01_Advanced_VirtualEnv.ipynb)** — PATH manipulation, site-packages internals, dependency conflict resolution, security attacks against package ecosystems.
